In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
car_data = pd.read_csv("datasets/car_price_prediction.csv")

In [4]:
car_data['Turbo'] = car_data['Engine volume'].str.contains('Turbo').astype(int)

car_data['Engine volume'] = (
    car_data['Engine volume']
    .str.replace(' Turbo', '')
    .astype(float)
)

car_data['Mileage'] = (
    car_data['Mileage']
    .str.replace(' km', '')
    .astype(int)
)

car_data['Levy'] = car_data['Levy'].replace('-', np.nan)
car_data['Levy'] = pd.to_numeric(car_data['Levy'])

In [5]:
X = car_data.drop(['Price', 'ID'], axis=1)
y = car_data['Price']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [7]:
model_counts = X_train['Model'].value_counts()

rare_models = model_counts[model_counts < 10].index

X_train['Model'] = X_train['Model'].replace(rare_models, 'Other')
X_test['Model'] = X_test['Model'].replace(rare_models, 'Other')

In [8]:
numeric_features = X_train.select_dtypes(include='number').columns
categorical_features = X_train.select_dtypes(exclude='number').columns

In [9]:
numerical_processor = SimpleImputer(strategy='median')

categorical_processor = OneHotEncoder(handle_unknown='ignore')

In [10]:
preprocessor = ColumnTransformer([
    ("numeric", numerical_processor, numeric_features),
    ("categorical", categorical_processor, categorical_features)
])

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [13]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(15389, 313)
(3848, 313)


In [15]:
model = LinearRegression()

In [16]:
model.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](313,)","[ 0., 0., 0.,...,-0.,-0., 0.]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.896e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,313


In [17]:
y_pred = model.predict(X_test_processed)

In [18]:
print(y_pred[:10])

[18957.82418337 18958.23775804 18956.83506365 18957.36977246
 18957.91975219 18957.87251774 18957.40837555 18957.52238747
 18957.64981239 18957.18705844]


In [21]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 12525.24884408961
RMSE: 17757.119444775944
R²: -0.011934264105095238


In [22]:
print(car_data['Price'].describe())

count    1.923700e+04
mean     1.855593e+04
std      1.905813e+05
min      1.000000e+00
25%      5.331000e+03
50%      1.317200e+04
75%      2.207500e+04
max      2.630750e+07
Name: Price, dtype: float64


In [23]:
print(car_data.nlargest(20, 'Price')[['Price', 'Manufacturer', 'Model', 'Prod. year']])

          Price   Manufacturer                    Model  Prod. year
16983  26307500           OPEL                    Combo        1999
8541     872946    LAMBORGHINI                     Urus        2019
1225     627220  MERCEDES-BENZ           G 65 AMG 63AMG        2020
5008     308906        PORSCHE                      911        2016
9367     297930  MERCEDES-BENZ                 AMG GT S        2015
14839    297930     LAND ROVER        Range Rover Vogue        2019
7749     288521            BMW  M5 Машина в максимально        2018
10759    260296          LEXUS                   LX 570        2018
5840     254024  MERCEDES-BENZ            GLE 400 A M G        2016
15283    250574  MERCEDES-BENZ                  GLE 400        2017
7283     228935  MERCEDES-BENZ               GLE 63 AMG        2018
2283     219527        BENTLEY           Continental GT        2012
7353     216391  MERCEDES-BENZ         G 65 AMG G63 AMG        2013
1145     194438  MERCEDES-BENZ                  

In [24]:
print(car_data.nsmallest(20, 'Price')[['Price', 'Manufacturer', 'Model', 'Prod. year']])

       Price   Manufacturer     Model  Prod. year
7815       1           OPEL     Astra        1999
16992      1      CHEVROLET   Lacetti        2006
221        3        HYUNDAI   Elantra        2011
753        3         NISSAN   X-Terra        2004
4776       3     VOLKSWAGEN     Jetta        2016
4958       3     VOLKSWAGEN     Jetta        2014
5890       3  MERCEDES-BENZ   CLK 230        2004
7276       3  MERCEDES-BENZ  G 55 AMG        2020
8993       3         TOYOTA   Prius C        2015
9730       3            KIA  Sportage        2015
10885      3      CHEVROLET     Cruze        2018
11636      3  MERCEDES-BENZ     C 220        1998
13419      3         TOYOTA   Prius C        2012
14492      3     VOLKSWAGEN     Jetta        2015
14642      3        PORSCHE  Panamera        2011
15347      3            BMW       525        1995
17596      3        HYUNDAI    Sonata        2011
1164       6         TOYOTA   Prius C        2015
5344       6         TOYOTA   Prius C        2015


In [25]:
price_z = (
    (car_data['Price'] - car_data['Price'].mean())
    / car_data['Price'].std()
)

outliers = car_data[price_z.abs() > 3]

print("Number of potential outliers:", len(outliers))
print(outliers[['Price', 'Manufacturer', 'Model', 'Prod. year']])

Number of potential outliers: 3
          Price   Manufacturer           Model  Prod. year
1225     627220  MERCEDES-BENZ  G 65 AMG 63AMG        2020
8541     872946    LAMBORGHINI            Urus        2019
16983  26307500           OPEL           Combo        1999


In [26]:
print(16983 in X_train.index)
print(16983 in X_test.index)

True
False


In [27]:
X_train = X_train.drop(index=16983)
y_train = y_train.drop(index=16983)

In [28]:
print(X_train.shape)
print(y_train.shape)

(15388, 17)
(15388,)


In [29]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [30]:
model.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](313,)","[ 0., 0., 0.,...,-0., 0., 0.]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.725e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,313


In [31]:
y_pred = model.predict(X_test_processed)

In [32]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 11976.136055051013
RMSE: 17650.633902906877
R²: 0.00016604001868347762


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 11976.136055051013
RMSE: 17650.633902906877
R²: 0.00016604001868347762
